# ONPE ERM2022 – Scraping Actas por Ubigeo
Notebook para extraer **organización política → votos (y porcentaje)** por **Ubigeo** desde:
`https://resultadoshistorico.onpe.gob.pe/ERM2022/EleccionesMunicipales/RePro`.

In [2]:

import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [3]:
BASE_URL = "https://resultadoshistorico.onpe.gob.pe/ERM2022/EleccionesMunicipales/RePro"
OUTPUT_CSV = "erm2022_distritos_selenium_sin_csv_entrada.csv"

SLEEP_BETWEEN_SELECTIONS = 0.7   # pausa para que cargue la página al cambiar combos


In [6]:
def crear_driver(headless=True):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver


def limpiar_numero(texto: str):
    if not texto:
        return None
    t = (
        texto.replace(",", "")
             .replace(".", "")
             .replace(" ", "")
    )
    return int(t) if t.isdigit() else None


def extraer_tabla_partidos(driver):
    """
    Tabla con cabecera:
    Organización política | Total | % Votos válidos | % Votos emitidos
    Devuelve lista de tuplas (organizacion_politica, total_votos)
    """
    wait = WebDriverWait(driver, 20)

    # Buscamos la tabla cuyo <th> dice "Organización política"
    tabla = wait.until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//table[.//th[contains(translate(., "
                "'óÓáÁéÉíÍúÚñÑ', 'oOaAeEiIuUnN'),"
                "'ORGANIZACION POLITICA')]]"
            )
        )
    )

    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});", tabla
    )

    filas = tabla.find_elements(By.XPATH, ".//tbody/tr")
    resultados = []

    for tr in filas:
        celdas = tr.find_elements(By.TAG_NAME, "td")
        if len(celdas) < 2:
            continue

        org = celdas[0].text.strip()
        total_txt = celdas[1].text.strip()

        if not org:
            continue

        up = org.upper()
        # si quieres incluir blancos/nulos/totales, comenta este bloque
        if any(x in up for x in [
            "TOTAL DE VOTOS VÁLIDOS",
            "TOTAL DE VOTOS EMITIDOS",
            "VOTOS EN BLANCO",
            "VOTOS NULOS",
        ]):
            continue

        total = limpiar_numero(total_txt)
        if total is None:
            continue

        resultados.append((org, total))

    return resultados


def ir_a_municipal_distrital(driver):
    """Intenta hacer clic en el menú 'Municipal distrital' > 'Resultados' (si existe)."""
    wait = WebDriverWait(driver, 20)
    try:
        # enlace lateral "Municipal distrital"
        link_md = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//a[contains(., 'Municipal distrital')]")
            )
        )
        link_md.click()
        time.sleep(0.5)

        # submenú "Resultados"
        link_res = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//a[contains(., 'Resultados')]")
            )
        )
        link_res.click()
        time.sleep(0.5)
    except Exception:
        # si no está ese menú, seguimos como estamos
        pass


def obtener_selects(driver):
    """
    Localiza los <select> de Departamento, Provincia y Distrito
    de forma robusta, usando las etiquetas <label>.
    """
    wait = WebDriverWait(driver, 20)

    dep_el = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//label[contains(., 'Departamento')]/following::select[1]")
        )
    )
    prov_el = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//label[contains(., 'Provincia')]/following::select[1]")
        )
    )
    dist_el = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//label[contains(., 'Distrito')]/following::select[1]")
        )
    )

    return Select(dep_el), Select(prov_el), Select(dist_el)

In [7]:
def main():
    driver = crear_driver(headless=True)
    registros = []

    try:
        driver.get(BASE_URL)
        ir_a_municipal_distrital(driver)

        dep_select, prov_select, dist_select = obtener_selects(driver)

        # Recorremos departamentos
        for dep_opt in dep_select.options:
            dep_val = dep_opt.get_attribute("value")
            dep_text = dep_opt.text.strip()

            # saltar opción "-- TODOS --" o vacía
            if not dep_val or "TODOS" in dep_text.upper():
                continue

            print(f"\nDepartamento: {dep_text}")
            dep_select.select_by_value(dep_val)
            time.sleep(SLEEP_BETWEEN_SELECTIONS)

            # tras cambiar departamento, refrescamos los selects
            dep_select, prov_select, dist_select = obtener_selects(driver)

            # Recorremos provincias del departamento
            for prov_opt in prov_select.options:
                prov_val = prov_opt.get_attribute("value")
                prov_text = prov_opt.text.strip()

                if not prov_val or "TODOS" in prov_text.upper():
                    continue

                print(f"  Provincia: {prov_text}")
                prov_select.select_by_value(prov_val)
                time.sleep(SLEEP_BETWEEN_SELECTIONS)

                # refrescar selects otra vez
                dep_select, prov_select, dist_select = obtener_selects(driver)

                # Recorremos distritos de la provincia
                for dist_opt in dist_select.options:
                    dist_val = dist_opt.get_attribute("value")
                    dist_text = dist_opt.text.strip()

                    if not dist_val or "TODOS" in dist_text.upper():
                        continue

                    ubigeo = dist_val  # normalmente aquí viene el ubigeo
                    print(f"    Distrito: {dist_text} (ubigeo {ubigeo})")

                    dist_select.select_by_value(dist_val)
                    time.sleep(SLEEP_BETWEEN_SELECTIONS)

                    # Extraer tabla para este distrito
                    try:
                        filas = extraer_tabla_partidos(driver)
                    except Exception as e:
                        print(f"      [ERROR tabla] {e}")
                        continue

                    for org, total in filas:
                        registros.append({
                            "ubigeo": ubigeo,
                            "departamento": dep_text,
                            "provincia": prov_text,
                            "distrito": dist_text,
                            "organizacion_politica": org,
                            "total_votos": total,
                        })

    finally:
        driver.quit()

    df = pd.DataFrame(registros)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\nListo -> {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

TimeoutException: Message: 
Stacktrace:
0   chromedriver                        0x00000001047fbaa4 cxxbridge1$str$ptr + 2943380
1   chromedriver                        0x00000001047f3760 cxxbridge1$str$ptr + 2909776
2   chromedriver                        0x000000010430a2bc _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74028
3   chromedriver                        0x00000001043518ac _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 366364
4   chromedriver                        0x0000000104392d78 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 633832
5   chromedriver                        0x0000000104345f10 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 318848
6   chromedriver                        0x00000001047bdb98 cxxbridge1$str$ptr + 2689672
7   chromedriver                        0x00000001047c13ac cxxbridge1$str$ptr + 2704028
8   chromedriver                        0x000000010479ea08 cxxbridge1$str